# Parquet in Depth

Covers `ParquetReader`, `ParquetSink`, `ParquetPipeline`, and `ParquetDataResource` with partitioned datasets, filter pushdown, column projection, PyArrow integration, and Dask-backed loads.

| # | Topic |
|---|---|
| 1 | ParquetReader — basic file and partitioned loads |
| 2 | Filter pushdown — dict-style and PyArrow-style filters |
| 3 | Column projection — reading a subset of columns |
| 4 | PyArrow integration — load_arrow and load_filtered_arrow |
| 5 | Dask integration — load_files for distributed DataFrames |
| 6 | ParquetSink — writing partitioned datasets with options |
| 7 | ParquetPipeline — materializing SQL sources to Parquet |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import datetime as dt
import tempfile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from boti_data import (
    DataHelper,
    ParquetPipeline,
    ParquetSink,
)
from boti_data.parquet import ParquetDataConfig, ParquetDataResource
from boti_data import ParquetReader


## 1. ParquetReader — basic file and partitioned loads

`ParquetReader` wraps `DataGateway` for Parquet backends. It can read single files or partitioned directories and supports both `pandas` and `dask` return types.

In [2]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "dataset"
    base.mkdir()

    # Seed single-file parquet
    df = pd.DataFrame({"id": [1, 2, 3], "name": ["Alice", "Bob", "Charlie"], "score": [95.0, 87.0, 72.0]})
    df.to_parquet(base / "data.parquet", index=False)

    with ParquetReader({"parquet_storage_path": str(base), "parquet_filename": "data"}) as reader:
        data = reader.load(return_type="pandas")
        print("Single file load:")
        print(data)


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "partitioned"
    (base / "year=2024").mkdir(parents=True)
    (base / "year=2025").mkdir(parents=True)

    pd.DataFrame({"id": [1, 2], "event": ["A", "B"]}).to_parquet(base / "year=2024" / "part.parquet", index=False)
    pd.DataFrame({"id": [3, 4], "event": ["C", "D"]}).to_parquet(base / "year=2025" / "part.parquet", index=False)

    with ParquetReader({"parquet_storage_path": str(base)}) as reader:
        data = reader.load(return_type="pandas")
        print("Partitioned dataset:")
        print(data)


In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "date_partitioned"
    for d in [dt.date(2024, 1, 1), dt.date(2024, 6, 15), dt.date(2025, 1, 1)]:
        p = base / f"partition_date={d.isoformat()}"
        p.mkdir(parents=True)
        pd.DataFrame({"id": [d.toordinal()], "value": [float(d.day)]}).to_parquet(p / "data.parquet", index=False)

    # Load only a date range using parquet_start_date / parquet_end_date
    with ParquetReader({
        "parquet_storage_path": str(base),
        "parquet_start_date": dt.date(2024, 1, 1),
        "parquet_end_date": dt.date(2024, 12, 31),
    }) as reader:
        data = reader.load(return_type="pandas")
        print("Date-range-filtered dataset:")
        print(data)
        print(f"\nRows: {len(data)}")


## 2. Filter pushdown — dict-style and PyArrow-style filters

`ParquetReader.load()` accepts dict-style filters that are pushed down to the parquet reader for predicate pushdown at the file/row-group level.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "filter_demo"
    base.mkdir()

    df = pd.DataFrame({
        "category": ["A", "A", "B", "B", "C"],
        "value": [10, 20, 30, 40, 50],
        "active": [True, False, True, False, True],
    })
    df.to_parquet(base / "data.parquet", index=False)

    with ParquetReader({"parquet_storage_path": str(base)}) as reader:
        # Dict-style filter
        filtered = reader.load(filters={"category": "A"}, return_type="pandas")
        print("category = A:")
        print(filtered)
        print()

        # Composite filter
        filtered2 = reader.load(filters={"value__gte": 30, "active": True}, return_type="pandas")
        print("value >= 30 and active = True:")
        print(filtered2)


The underlying `ParquetDataResource` also exposes `load_filtered()` (dict-style) and `load_files()` (PyArrow list-style), useful for direct PyArrow filter expressions.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "pyarrow_filters"
    base.mkdir()

    df = pd.DataFrame({"group": ["x", "x", "y", "z"], "val": [1.0, 2.0, 3.0, 4.0]})
    df.to_parquet(base / "data.parquet", index=False)

    config = ParquetDataConfig(
        parquet_storage_path=str(base),
        parquet_filename="data",
    )
    with ParquetDataResource(config) as resource:
        # PyArrow list-style filters for predicate pushdown
        pa_filters = [("group", "=", "x")]
        result_pa = resource.load_files(filters=pa_filters).compute()
        print("PyArrow-style filter (group = x):")
        print(result_pa)
        print()

        # Dict-style filter via load_filtered
        result_dict = resource.load_filtered(filters={"val__gte": 2.0}).compute()
        print("Dict-style filter (val >= 2.0):")
        print(result_dict)


## 3. Column projection — reading a subset of columns

Projecting only the needed columns reduces I/O and memory, especially in columnar formats like Parquet.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "projection"
    base.mkdir()

    wide = pd.DataFrame({
        "id": range(100),
        "name": [f"user_{i}" for i in range(100)],
        "email": [f"user_{i}@example.com" for i in range(100)],
        "address": [f"addr_{i}" for i in range(100)],
        "score": [float(i) for i in range(100)],
    })
    wide.to_parquet(base / "wide.parquet", index=False)

    with ParquetReader({"parquet_storage_path": str(base), "parquet_filename": "wide"}) as reader:
        # Load all columns
        all_cols = reader.load(return_type="pandas")
        print(f"All columns ({len(all_cols.columns)}): {list(all_cols.columns)}")

        # Load only id and score via column kwarg in options
        projected = reader.load(return_type="pandas", columns=["id", "score"])
        print(f"Projected columns ({len(projected.columns)}): {list(projected.columns)}")
        print(projected.head())


## 4. PyArrow integration — load_arrow and load_filtered_arrow

`ParquetDataResource` works directly with PyArrow tables for zero-copy columnar access.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "arrow_demo"
    base.mkdir()

    df = pd.DataFrame({"x": [1, 2, 3], "y": [10.0, 20.0, 30.0], "label": ["a", "b", "c"]})
    df.to_parquet(base / "data.parquet", index=False)

    config = ParquetDataConfig(
        parquet_storage_path=str(base),
        parquet_filename="data",
    )
    with ParquetDataResource(config) as resource:
        table = resource.load_arrow(columns=["x", "y"])
        print(f"PyArrow table: {type(table).__name__}")
        print(f"Columns: {table.column_names}")
        print(f"Num rows: {table.num_rows}")
        print(f"Schema: {table.schema}")
        print()
        print(table.to_pandas())


`load_filtered_arrow()` combines predicate pushdown with PyArrow output.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "arrow_filter"
    base.mkdir()

    pd.DataFrame({"a": range(10), "b": [f"v{i}" for i in range(10)]}).to_parquet(base / "data.parquet", index=False)

    config = ParquetDataConfig(parquet_storage_path=str(base))
    with ParquetDataResource(config) as resource:
        table = resource.load_filtered_arrow({"a__gte": 5})
        print("Filtered arrow table (a >= 5):")
        print(table.to_pandas())


## 5. Dask integration — load_files for distributed DataFrames

`load_files()` returns a Dask DataFrame, enabling lazy evaluation and distributed computation.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "dask_demo"
    base.mkdir()

    # Write multiple parquet files for multi-partition reading
    for i in range(3):
        part = pd.DataFrame({"id": range(i * 10, (i + 1) * 10), "value": [float(j) for j in range(i * 10, (i + 1) * 10)]})
        part.to_parquet(base / f"part_{i}.parquet", index=False)

    config = ParquetDataConfig(parquet_storage_path=str(base))
    with ParquetDataResource(config) as resource:
        ddf = resource.load_files()
        print(f"Dask DataFrame type: {type(ddf).__name__}")
        print(f"Npartitions: {ddf.npartitions}")
        print(f"Divisions: {ddf.divisions}")
        print()

        # Trigger compute
        result = ddf.compute()
        print("Computed result:")
        print(result)


In [ ]:
# Reading via DataHelper with return_type="dask" also yields a Dask DataFrame
with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp) / "helper_dask"
    base.mkdir()

    pd.DataFrame({"id": [1, 2], "val": [10, 20]}).to_parquet(base / "data.parquet", index=False)

    helper = DataHelper(
        backend="parquet",
        storage_path=str(base),
        df_params={"return_type": "dask"},
    )
    ddf = helper.load()
    print(f"DataHelper dask load: {type(ddf).__name__}, npartitions={ddf.npartitions}")
    print(ddf.compute())


## 6. ParquetSink — writing partitioned datasets with options

`ParquetSink` writes DataFrames to partitioned or non-partitioned Parquet datasets.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "sink_output"

    with ParquetSink({"parquet_storage_path": str(out)}, partition_on=None) as sink:
        df = pd.DataFrame({
            "city": ["NYC", "NYC", "LA", "LA", "CHI"],
            "value": [100, 200, 300, 400, 500],
            "date": pd.to_datetime(["2024-01-01", "2024-01-02", "2024-01-01", "2024-01-02", "2024-01-01"]),
        })
        result = sink.write(df)

    print(f"Write result path: {result.path}")
    print(f"Files: {list(Path(result.path).iterdir())}")

    # Read it back
    with ParquetReader({"parquet_storage_path": result.path}) as reader:
        print(f"\nRead back:\n{reader.load(return_type="pandas")}")


In [ ]:
# Partitioned write: one directory per partition value
with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "partitioned_sink"

    with ParquetSink({"parquet_storage_path": str(out)}, partition_on=["city"]) as sink:
        df = pd.DataFrame({
            "city": ["NYC", "LA", "NYC", "LA"],
            "sales": [1000, 2000, 1500, 2500],
            "month": [1, 1, 2, 2],
        })
        result = sink.write(df)

    print(f"Partitioned output: {result.path}")
    for p in sorted(Path(result.path).iterdir()):
        print(f"  {p.name}/")
        for f in sorted(p.iterdir()):
            print(f"    {f.name}")

    # Reading back automatically resolves partitions
    with ParquetReader({"parquet_storage_path": result.path}) as reader:
        print(f"\nRead back:\n{reader.load(return_type="pandas")}")


`ParquetSink` also accepts a `destination` that is a `ParquetReader`, allowing direct chaining.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "chained"

    with ParquetReader({"parquet_storage_path": str(out)}) as reader:
        with ParquetSink(reader, partition_on=None) as sink:
            df = pd.DataFrame({"x": [1, 2, 3], "y": [4.0, 5.0, 6.0]})
            sink.write(df)

            # Read back via the same reader
            result = reader.load(return_type="pandas")
            print("Chained sink -> reader load:")
            print(result)


## 7. ParquetPipeline — materializing SQL sources to Parquet

`ParquetPipeline` connects a SQL `DataHelper` source to a Parquet `destination`, optionally re-reading the materialized data for a second pass.

In [ ]:
from sqlalchemy import Date, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

class Base(DeclarativeBase):
    pass

class Event(Base):
    __tablename__ = "events"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))

with tempfile.TemporaryDirectory() as tmp:
    db_path = Path(tmp) / "pipeline.db"
    engine = create_engine(f"sqlite:///{db_path}")

    Base.metadata.create_all(engine)
    with Session(engine) as session:
        session.add_all([
            Event(id=1, event_date=dt.date(2026, 1, 15), status="active"),
            Event(id=2, event_date=dt.date(2026, 2, 1), status="inactive"),
            Event(id=3, event_date=dt.date(2026, 3, 10), status="active"),
        ])
        session.commit()
    engine.dispose()

    helper = DataHelper(
        backend="sqlalchemy",
        connection_url=f"sqlite:///{db_path}",
        poolclass="sqlalchemy.pool.NullPool",
        query_only=False,
        table="events",
    )

    pipeline = ParquetPipeline(
        helper,
        {"parquet_storage_path": str(Path(tmp) / "events_dataset")},
        date_field="event_date",
        partition_on=["status"],
    )

    materialized = pipeline.materialize()
    print(f"Materialized path: {materialized.path}")
    print(f"Reloaded flag: {materialized.reloaded}")

    # Reload from the materialized Parquet with additional filters
    reloaded = pipeline.materialize(reload=True, reload_options={"filters": {"status": "active"}})
    if reloaded.frame is not None:
        print(f"\nReloaded (active only): {reloaded.frame.compute() if hasattr(reloaded.frame, "compute") else reloaded.frame}")
    pipeline.close()


### Summary

- **`ParquetReader`** — gateway-based reader with dict-style filters, column projection, and `pandas`/`dask` return types.
- **`ParquetDataResource`** — low-level resource with PyArrow (`load_arrow`, `load_filtered_arrow`) and Dask (`load_files`) methods.
- **`ParquetSink`** — writes DataFrames as partitioned or unpartitioned Parquet datasets; accepts a reader as destination for chaining.
- **`ParquetPipeline`** — materializes SQL sources to Parquet with optional re-reading for efficient second-pass queries.
- **Filter pushdown** — both dict-style (`{"col__gte": val}`) and PyArrow list-style (`[("col", "=", val)]`) filters are supported and pushed to the parquet reader.